## Ensembles: Random Forest, Gradient Boosting en XGBoost

**Wat is een ensemble?**
Een ensemble combineert de voorspellingen van meerdere modellen tot één voorspelling. Eén decision tree maakt snel fouten en overfit makkelijk, maar als je veel verschillende bomen combineert, middelen de fouten elkaar uit.

Er zijn twee hoofdtechnieken:
- **Bagging** (Random Forest): veel bomen worden onafhankelijk van elkaar getraind, elk op een willekeurige steekproef van de data (bootstrap) en met een willekeurige selectie van features. De voorspelling is de meerderheid van de stemmen. Dit verlaagt vooral de **variance**.
- **Boosting** (Gradient Boosting, XGBoost): bomen worden na elkaar getraind. Elke nieuwe boom probeert de fouten van de vorige bomen te corrigeren. Dit verlaagt vooral de **bias**.

**Waarom ensembles voor deze dataset?**
Decision trees kunnen niet-lineaire verbanden en interacties oppikken, zoals het effect van een hoge `avg_glucose_level` of de combinatie van hoge leeftijd en hoge bloeddruk. Logistic regression kan dit niet.

Ensembles op basis van decision trees hebben geen standaardisatie nodig: een boom splitst op één feature tegelijk, waardoor de schaal geen invloed heeft.

### Random Forest: baseline

We beginnen met een Random Forest met standaardinstellingen en 300 bomen. We evalueren op dezelfde manier als bij logistic regression: 5-fold `StratifiedKFold`.

In [ ]:
"""Ensembles voor de Kaggle-competitie Predicting stroke."""
import sys

sys.path.append("..")  # zodat de map src gevonden wordt

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.preprocessing import StrokeData

data = StrokeData()
X, y = data.get_xy()
X_test = data.get_test()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
}

rf_default = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

scores = cross_validate(rf_default, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.000
recall: 0.000
f1: 0.000


**Observatie:** net als de baseline van logistic regression voorspelt het Random Forest voor geen enkele patiënt een herseninfarct. Bijna alle bladeren van de bomen bevatten vooral patiënten zonder herseninfarct, waardoor de meerderheid van de 300 bomen altijd "geen herseninfarct" stemt. Een krachtiger model lost de onbalans dus niet vanzelf op.

### Class weights

Bij logistic regression loste `class_weight="balanced"` het probleem direct op. We proberen hetzelfde bij het Random Forest: een patiënt met een herseninfarct telt dan zwaarder mee bij het kiezen van de splits en bij het stemmen in de bladeren.

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)

scores = cross_validate(rf_balanced, X, y, cv=cv, scoring=scoring)
for metric in scoring:
    print(f"{metric}: {scores[f'test_{metric}'].mean():.3f}")

precision: 0.080
recall: 0.021
f1: 0.033


**Observatie:** anders dan bij logistic regression helpt `class_weight="balanced"` hier nauwelijks: de recall is slechts 0,021. De oorzaak is overfitting. Standaard groeit elke boom door tot alle bladeren zuiver zijn (alleen patiënten van één klasse). De bomen leren de trainingsdata dan uit het hoofd. In een zuiver blad valt er niets te wegen, dus de class weights hebben geen effect. Nieuwe patiënten met een herseninfarct komen vrijwel altijd terecht in een blad met "geen herseninfarct".

### Overfitting beperken met `min_samples_leaf`

Met `min_samples_leaf` bepalen we hoeveel patiënten er minimaal in een blad moeten zitten. Een hogere waarde maakt de bomen eenvoudiger: de bladeren blijven gemengd, waardoor de class weights effect hebben. We vergelijken een aantal waarden.

In [ ]:
rows = []
for leaf in [1, 5, 20, 50, 100]:
    rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=leaf, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    scores = cross_validate(rf, X, y, cv=cv, scoring=scoring)
    rows.append({
        "min_samples_leaf": leaf,
        "precision": scores["test_precision"].mean(),
        "recall": scores["test_recall"].mean(),
        "f1": scores["test_f1"].mean(),
    })

pd.DataFrame(rows).round(3)

,min_samples_leaf,precision,recall,f1
0,1,0.080,0.021,0.033
1,5,0.073,0.306,0.118
2,20,0.060,0.681,0.110
3,50,0.052,0.772,0.097
4,100,0.048,0.816,0.091


**Observatie:** hoe groter `min_samples_leaf`, hoe hoger de recall (van 0,021 naar 0,816) en hoe lager de precision. De bladeren blijven gemengd, waardoor de class weights nu wel effect hebben. Dit is de bias-variance tradeoff: kleine bladeren geven complexe bomen die overfitten (hoge variance), grote bladeren geven eenvoudigere bomen die beter generaliseren. Bij de standaard threshold van 0,5 is de F1-score het hoogst bij `min_samples_leaf = 5` (0,118). Deze vergelijking is echter niet eerlijk: elk model heeft een andere optimale threshold.

### Threshold per model

Net als bij logistic regression zoeken we per model de threshold met de hoogste F1-score. Pas daarna kunnen we de waarden van `min_samples_leaf` eerlijk vergelijken.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import cross_val_predict

thresholds = np.arange(0.05, 0.96, 0.05)

rows = []
for leaf in [5, 20, 50, 100]:
    rf = RandomForestClassifier(
        n_estimators=300, min_samples_leaf=leaf, class_weight="balanced",
        random_state=42, n_jobs=-1,
    )
    y_proba = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]
    f1_scores = [f1_score(y, (y_proba >= t).astype(int)) for t in thresholds]
    best_threshold = thresholds[int(np.argmax(f1_scores))]
    y_pred = (y_proba >= best_threshold).astype(int)
    rows.append({
        "min_samples_leaf": leaf,
        "beste_threshold": round(best_threshold, 2),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1": max(f1_scores),
    })

pd.DataFrame(rows).round(3)

,min_samples_leaf,beste_threshold,precision,recall,f1
0,5,0.45,0.072,0.397,0.121
1,20,0.65,0.076,0.484,0.132
2,50,0.75,0.086,0.342,0.137
3,100,0.75,0.084,0.335,0.134


**Observatie:** met een eigen threshold per model verandert de volgorde volledig. `min_samples_leaf = 5` scoorde bij threshold 0,5 het best, maar is nu het slechtst (0,121). De waarden 20, 50 en 100 liggen dicht bij elkaar (0,132–0,137). Dit laat zien dat modellen alleen eerlijk te vergelijken zijn als de threshold per model is afgestemd. We kiezen `min_samples_leaf = 50`: de hoogste F1-score (0,137) met eenvoudige bomen die minder snel overfitten. Geen van de modellen haalt de 0,143 van logistic regression.